# Transfer learning

For an example of transfer learning in TensorFlow for a computer vision task, let's use a 
1. pre-trained model from TensorFlow's Keras Applications, 
2. fine-tune it on a smaller dataset, 
3. improve its performance for a specific task. 
4. 
5. This example will demonstrate how to adapt a **`MobileNetV2 model`**, **pre-trained on ImageNet**, for a new classification task using the `CIFAR-10 dataset—a collection of 60,000 32x32 color images in 10 different classes.`

### Step 1: Import Libraries and Load Data

In [3]:
!pip install tensorflow

  Using cached tensorflow-2.21.0-cp310-cp310-win_amd64.whl.metadata (4.5 kB)
  Using cached absl_py-2.4.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached protobuf-7.34.1-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached requests-2.33.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached termcolor-3.3.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached wrapt-2.1.2-cp310-cp310-win_amd64.whl.metadata (7.6 kB)
  Using cached grpcio-1.80.0-cp310-cp310-win_amd64.whl.metadata (3.9 kB)
  Using cached keras-3.12.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached h5py-3

In [4]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Load CIFAR-10 data
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# Normalize pixel values and convert labels to one-hot encoding
x_train, x_test = x_train / 255.0, x_test / 255.0
y_train, y_test = to_categorical(y_train, 10), to_categorical(y_test, 10)

 62291968/170498071 ━━━━━━━━━━━━━━━━━━━━ 7:30:10 250us/step

Exception: URL fetch failure on https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz: None -- retrieval incomplete: got only 62287899 out of 170498071 bytes

### `Data Augmentation:`

Data augmentation is a strategy used in machine learning to increase the diversity and amount of training data without actually collecting new data. This is achieved by creating modified versions of the existing data. For example, in image data, augmentation techniques might include rotation, scaling, flipping, cropping, or adding noise.

Data augmentation is beneficial for several reasons:

1. **`Preventing Overfitting`**: By providing more varied data, the model is less likely to memorize the training data and more likely to generalize to new, unseen data.
2. **`Improving Model Performance`**: More data generally leads to better model performance, especially in deep learning.
3. **`Dealing with Imbalanced Data`**: Augmentation can help balance the dataset by creating more instances of under-represented classes.

Remember, the type of augmentation should make sense for the problem at hand. For example, flipping an image might be fine for a general object recognition task, but not for a task where `orientation is crucial, like reading numbers` or letters.

### Step 2: Preprocess Data

Since MobileNetV2 expects input images of size `224x224`, and CIFAR-10 images are `32x32`, we need to resize them. We'll also apply some data augmentation:


In [ ]:
# Data Augmentation
datagen = ImageDataGenerator(
    featurewise_center=False,
    samplewise_center=False,
    featurewise_std_normalization=False,
    samplewise_std_normalization=False,
    zca_whitening=False,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    vertical_flip=False,
    zoom_range=0.1,
    fill_mode='nearest')

datagen.fit(x_train)

### Step 3: Modify Pre-trained Model

We'll load `MobileNetV2` without its top layer (since we're applying it to a new task with 10 classes instead of 1000), add a new classifier on top, and freeze the layers of MobileNetV2:

**We use less classes than 1000**

In [ ]:
# Load MobileNetV2 without the top layer
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(32, 32, 3))

# Freeze the base_model
base_model.trainable = False 

# Add custom layers on top for our task
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)  # New FC layer, random init
predictions = Dense(10, activation='softmax')(x)  # New softmax layer

# Define the model
model = Model(inputs=base_model.input, outputs=predictions)

### Step 4: Compile and Train the Model

In [ ]:
%%time
model.compile(optimizer=tf.keras.optimizers.Adam(), 
loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(datagen.flow(x_train, y_train, batch_size=32), 
                    steps_per_epoch=len(x_train) / 32, epochs=10,
                    validation_data=(x_test, y_test), verbose=1)

In [ ]:
import matplotlib.pyplot as plt
# Plot training & validation accuracy values
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Test'], loc='upper left')
plt.show()

# Plot training & validation loss values
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Test'], loc='upper left')
plt.show()

# An extra run with more epochs and Increased Batch Size

In [ ]:
%%time
#early stopping
from tensorflow.keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
model.compile(optimizer=tf.keras.optimizers.legacy.Adam(), 
loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(datagen.flow(x_train, y_train, batch_size=64), 
                    steps_per_epoch=len(x_train) / 64, epochs=50,
                    validation_data=(x_test, y_test), verbose=1, callbacks=[early_stopping])
import matplotlib.pyplot as plt
# Plot training & validation accuracy values
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Test'], loc='upper left')
plt.show()

# Plot training & validation loss values
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Test'], loc='upper left')
plt.show()

### Step 5: Fine-Tuning (Optional)

After the initial training with the top layers, you can choose to unfreeze some of the bottom layers of the MobileNetV2 model and `continue training to improve accuracy`. Before doing so, it's important to recompile the model:

In [ ]:
# Unfreeze some layers in the base model
base_model.trainable = True
fine_tune_at = 100  # This is the number of layers from the top to freeze
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# Recompile the model
model.compile(optimizer=tf.keras.optimizers.Adam(lr=0.0001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Continue training
history_fine = model.fit(datagen.flow(x_train, y_train, batch_size=32),
                         steps_per_epoch=len(x_train) / 32, epochs=5,
                         validation_data=(x_test, y_test), verbose=1)

This example demonstrates the basics of applying transfer learning with TensorFlow to improve performance on a computer vision task using a smaller dataset. Fine-tuning and data augmentation are powerful techniques to increase accuracy further and adapt the pre-trained model to the new task more effectively.

In [ ]:
import matplotlib.pyplot as plt

def plot_history(histories, key='accuracy'):
    plt.figure(figsize=(16, 4))
    
    for name, history in histories:
        val = plt.plot(history.epoch, history.history['val_'+key],
                       '--', label=name.title()+' Val')
        plt.plot(history.epoch, history.history[key], color=val[0].get_color(),
                 label=name.title()+' Train')

    plt.xlabel('Epochs')
    plt.ylabel(key.replace('_', ' ').title())
    plt.legend()
    plt.xlim([0, max(history.epoch)])

# Plot accuracy
plot_history([('Pre Fine-Tuning', history),
              ('Fine-Tuning', history_fine)],
             key='accuracy')

# Plot loss
plot_history([('Pre Fine-Tuning', history),
              ('Fine-Tuning', history_fine)],
             key='loss')